# Parallel Computing Lab
## Sequential vs Multiprocessing

Three problems are solved in two ways and their timings are compared :

1. Matrix Multiplication (500 x 500)
2. Odd / Even Number Counting
3. Factorial of a Big Number




In [ ]:
import time
import matplotlib.pyplot as plt
from multiprocessing import cpu_count

import matmul
import oddeven
import factorial

workers = cpu_count()
print('CPU cores available :', workers)

# we will keep all the timings here to draw the graphs later
results = {}


---
## 1. Matrix Multiplication

Two random 500 x 500 matrices are multiplied.


In [ ]:
A = matmul.make_matrix(matmul.N)
B = matmul.make_matrix(matmul.N)
print('Matrix size :', matmul.N, 'x', matmul.N)

t1 = time.time()
C1 = matmul.sequential(A, B)
seq = time.time() - t1

t2 = time.time()
C2 = matmul.parallel(A, B, workers)
par = time.time() - t2

assert C1 == C2, 'results do not match!'
results['Matrix Mult.'] = [seq, par]

print('Sequential time      : %.2f sec' % seq)
print('Multiprocessing time : %.2f sec' % par)
print('Speedup              : %.2fx' % (seq / par))


---
## 2. Odd / Even Counting

Counts how many odd and even numbers are there from 0 to N.
In the parallel version the **range of numbers** is divided among the processes.


In [ ]:
print('Total numbers :', oddeven.N)

t1 = time.time()
r1 = oddeven.sequential(oddeven.N)
seq = time.time() - t1

t2 = time.time()
r2 = oddeven.parallel(oddeven.N, workers)
par = time.time() - t2

assert r1 == r2, 'results do not match!'
results['Odd / Even'] = [seq, par]

print('Odd numbers  :', r1[0])
print('Even numbers :', r1[1])
print()
print('Sequential time      : %.2f sec' % seq)
print('Multiprocessing time : %.2f sec' % par)
print('Speedup              : %.2fx' % (seq / par))


---
## 3. Factorial of a Big Number

Factorial is just `1 * 2 * 3 * ... * N`.
So in the parallel version the range `1..N` is divided among the processes,
each process finds its own **partial product**, and at the end all the partial
products are multiplied together to get the final answer.


In [ ]:
print('Finding factorial of :', factorial.N)

t1 = time.time()
f1 = factorial.sequential(factorial.N)
seq = time.time() - t1

t2 = time.time()
f2 = factorial.parallel(factorial.N, workers)
par = time.time() - t2

assert f1 == f2, 'results do not match!'
results['Factorial'] = [seq, par]

# the answer is too big to print, so we only show how many digits it has
digits = int(f1.bit_length() * 0.30103) + 1
print('Digits in the answer :', digits)
print()
print('Sequential time      : %.2f sec' % seq)
print('Multiprocessing time : %.2f sec' % par)
print('Speedup              : %.2fx' % (seq / par))


---
## Summary Table

Speedup = Sequential Time / Parallel Time  
Efficiency = ( Speedup / Number of Cores ) x 100


In [ ]:
names = list(results.keys())
seq_times = [results[n][0] for n in names]
par_times = [results[n][1] for n in names]
speedup = [s / p for s, p in zip(seq_times, par_times)]
efficiency = [s / workers * 100 for s in speedup]

print('%-22s %12s %12s %10s %12s' % ('Problem', 'Sequential', 'Parallel', 'Speedup', 'Efficiency'))
print('-' * 72)
for i in range(len(names)):
    print('%-22s %10.2f s %10.2f s %9.2fx %10.2f %%' % (
        names[i], seq_times[i], par_times[i], speedup[i], efficiency[i]))


---
## Graphs


### Graph 1 : Execution Time


In [ ]:
x = range(len(names))
width = 0.35

plt.figure(figsize=(8, 5))
b1 = plt.bar([i - width/2 for i in x], seq_times, width, label='Sequential', color='#0072B2')
b2 = plt.bar([i + width/2 for i in x], par_times, width, label='Multiprocessing', color='#E69F00')

plt.bar_label(b1, fmt='%.1f')
plt.bar_label(b2, fmt='%.1f')

plt.xticks(list(x), names)
plt.ylabel('Time (seconds)')
plt.title('Execution Time : Sequential vs Multiprocessing')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()


### Graph 2 : Speedup


In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.bar(names, speedup, color='#0072B2', width=0.5)
plt.bar_label(bars, fmt='%.2fx')

# a line at 1x, below this line parallel would be slower than sequential
plt.axhline(1, color='gray', linestyle='--', linewidth=1)

plt.ylabel('Speedup (times faster)')
plt.title('Speedup achieved by Multiprocessing')
plt.grid(axis='y', alpha=0.3)
plt.show()


### Graph 3 : Efficiency


In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.bar(names, efficiency, color='#0072B2', width=0.5)
plt.bar_label(bars, fmt='%.1f%%')

plt.ylabel('Efficiency (%)')
plt.ylim(0, 100)
plt.title('Efficiency of Multiprocessing on %d cores' % workers)
plt.grid(axis='y', alpha=0.3)
plt.show()
